In [ ]:
!pip install huggingface_hub pyngrok openai-whisper nest_asyncio

In [ ]:
!ngrok config add-authtoken 2f2i5s0cdMFS65gx7UDpaLYyieJ_73i1yKGHQNjVbzouaYPex

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Libraries**

In [ ]:
# AI & Deep Learning
from sentence_transformers import SentenceTransformer
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, AutoModelForCTC, Wav2Vec2Processor
from transformers import pipeline, AutoModelForSpeechSeq2Seq, WhisperProcessor
from peft import PeftModel
import whisper
import torch

# Network
import asyncio
import websockets
from pyngrok import ngrok

# Utilities
import numpy as np
import time
import json
import joblib
import functools
import struct

**Configurations and Variables**

In [ ]:
# --- CONFIGURATION ---
VAD_SAMPLE_RATE = 16000
VAD_WINDOW = 512
SILENCE_THRESHOLD = 0.3
SILENCE_CHUNKS = int(SILENCE_THRESHOLD / (VAD_WINDOW / VAD_SAMPLE_RATE))
MIN_SENTENCE_LENGTH = 1.0
MAX_SENTENCE_LENGTH = 5.0
PORT = 5001

# --- GLOBAL MODEL VARIABLES ---
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"
dtype=torch.float16 if use_fp16 else torch.float32
whisper_model = None
wav2vec_en_processor = None
wav2vec_en = None
wav2vec_vn_processor = None
wav2vec_vn = None
finetune_translator = None
sentence_model = None
kmeans = None
vad_model = None
finetuned_translator, sentence_model, kmeans = None                                                                 # translator
wav2vec_en_processor, wav2vec_en, wav2vec_vn_processor, wav2vec_vn = None                                       # wav2vec
finetuned_whisper_vie_processor, finetuned_whisper_vie, finetuned_whisper_en_processor, finetuned_whisper_en = None   # whisper
base_whisper_tiny, whisper_tiny_base_pipe, whisper_largev3, whisper_largev3_pipe = None

# --- PATH VARIABLES ---
kmeans_path = "/content/drive/MyDrive/PBL6/kmeans.joblib"
nmt_path = "/content/drive/MyDrive/PBL6/finetune_mbart"
sentence_model_path = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
wav2vec_en_path = "patrickvonplaten/wav2vec2-base-timit-demo-colab"
wav2vec_vn_path = "pdabo1607/Vietnamese_Wav2Vec_Finetune_round2"
BASE_WHISPER_MODEL = "openai/whisper-tiny"
WHISPER_LARGE = "openai/whisper-large-v3"
VIE_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/whisper_tiny_vi"
EN_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/whisper_tiny_en"

**Load models**

In [ ]:
# --- LOAD MODELS ---
def load_models():
    print("[Init] Đang load models... Vui lòng đợi.")
    print(f"[Init] Device: {device}")

    # Load models
    try:
        base_whisper_tiny = AutoModelForSpeechSeq2Seq.from_pretrained(
                        BASE_WHISPER_MODEL, torch_dtype=dtype, device_map=device
                    )
        whisper_tiny_base_pipe = pipeline("automatic-speech-recognition", model=BASE_WHISPER_MODEL)
        print("[Init] Base Whisper Tiny Loaded")

        finetuned_whisper_vie= PeftModel.from_pretrained(base_whisper_tiny, VIE_ADAPTER_WHISPER_PATH)
        finetuned_whisper_vie_processor = WhisperProcessor.from_pretrained(VIE_ADAPTER_WHISPER_PATH)
        finetuned_whisper_vie = finetuned_whisper_vie.merge_and_unload()
        print("[Init] Whisper Vie Loaded")

        finetuned_whisper_en= PeftModel.from_pretrained(base_whisper_tiny, EN_ADAPTER_WHISPER_PATH)
        finetuned_whisper_en_processor = WhisperProcessor.from_pretrained(EN_ADAPTER_WHISPER_PATH)
        finetuned_whisper_en = finetuned_whisper_en.merge_and_unload()
        print("[Init] Whisper En Loaded")

        whisper_largev3_pipe = pipeline("automatic-speech-recognition", model=WHISPER_LARGE)
        print("[Init] Whisper Large Loaded")

        wav2vec_en_processor = Wav2Vec2Processor.from_pretrained(wav2vec_en_path)
        wav2vec_en = AutoModelForCTC.from_pretrained(wav2vec_en_path).to(device)
        print("[Init] Wav2Vec2_en Loaded")


        wav2vec_vn_processor = Wav2Vec2Processor.from_pretrained(wav2vec_vn_path)
        wav2vec_vn = AutoModelForCTC.from_pretrained(wav2vec_vn_path).to(device)
        print("[Init] Wav2Vec2_vn Loaded")

        trans_model = MBartForConditionalGeneration.from_pretrained(nmt_path)
        tokenizer = MBart50TokenizerFast.from_pretrained(nmt_path)
        finetune_translator = pipeline(
            "translation",
            model=trans_model,
            tokenizer=tokenizer,
            device=0 if device == "cuda" else -1
        )
        print("[Init] mbart Loaded")

        sentence_model = SentenceTransformer(sentence_model_path)
        print("[Init] Sentence Model Loaded")

        kmeans = joblib.load(kmeans_path)
        print("[Init] Kmeans Loaded")

        vad_model, _ = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                  model='silero_vad',
                                  force_reload=False)
        print("[Init] VAD Model Loaded")
    except Exception as e:
        print(f"[Init Warning] Không thể load model dịch (kiểm tra lại đường dẫn): {e}")
        # Fallback hoặc xử lý lỗi tùy ý
        finetune_translator = None

**Helper Functions**

In [ ]:
def translate(translator, text, is_en):
    # Defined domain of sentence
    domain = kmeans.predict(sentence_model.encode([text]))[0]
    dtext = f"<D{domain}> " + text

    if is_en:
        src_lang = "en_XX"
        tgt_lang = "vi_VN"
    else:
        src_lang = "vi_VN"
        tgt_lang = "en_XX"

    # Get translate result
    result = translator(
        text,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        max_length=128
    )

    return result[0]['translation_text']

In [ ]:
def wav2vec_transcribe(audio_np, origin_lang):
    # Transcription audio
    transcription = (wav2vec_en_processor.decode(
                      torch.argmax(
                        wav2vec_en(
                          wav2vec_en_processor(
                            audio_np,
                            sampling_rate=16000,
                            return_tensors="pt",
                            padding=True
                        ).input_values.to(device)).logits, dim=-1)[0])
                     if origin_lang == 0 else
                     wav2vec_vn_processor.decode(
                        torch.argmax(
                          wav2vec_vn(
                            wav2vec_vn_processor(
                              audio_np,
                              sampling_rate=16000,
                              return_tensors="pt",
                              padding=True
                          ).input_values.to(device)).logits, dim=-1)[0]))
    text = transcription.strip()
    return text if text else None

In [ ]:
def whisper_transcribe(audio_np, origin_lang, model_name):
    result = None
    
    if model_name == "tiny":
        result = whisper_tiny_base_pipe(
            audio_np,
            generate_kwargs={"language": "en" if origin_lang == 0 else "vi", "task": "transcribe"}
        )

    elif model_name == "large":
        result = whisper_largev3_pipe(
            audio_np,
            generate_kwargs={"language": "en" if origin_lang == 0 else "vi", "task": "transcribe"}
        )

    elif model_name == "tiny_finetuned":
        if origin_lang == 0:
            inputs = finetuned_whisper_en_processor(audio_np, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device).to(dtype)

            with torch.no_grad():
                predicted_ids = finetuned_whisper_en.generate(
                    input_features,
                    language="en",      # Sets the language
                    task="transcribe",  # Sets the task
                    max_new_tokens=225
                )

            # Decode
            result = finetuned_whisper_en_processor.batch_decode(predicted_ids, skip_special_tokens=True)
            return result[0] if result else None
            
        elif origin_lang == 1:
            inputs = finetuned_whisper_vie_processor(audio_np, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device).to(dtype)

            with torch.no_grad():
                predicted_ids = finetuned_whisper_vie.generate(
                    input_features,
                    language="vi",      # Sets the language
                    task="transcribe",  # Sets the task
                    max_new_tokens=225
                )

            # Decode
            result = finetuned_whisper_vie_processor.batch_decode(predicted_ids, skip_special_tokens=True)
            return result[0] if result else None
        else:
            return None

    if result:
        transcript = result["text"]
        return transcript
    return None

**Main Logics**

In [ ]:
# --- ASR + TRANSLATION LOGIC ---
def run_inference_sync(audio_np, start_offset, duration, startClock, origin_lang, target_lang, transcription_model, translation_model="mbart"):
    try:
        # Transcribe based on model selection
        if transcription_model.startswith("whisper"):
            text = whisper_transcribe(audio_np, origin_lang, transcription_model)
        elif transcription_model.startswith("wav2vec"):
            text = wav2vec_transcribe(audio_np, origin_lang)
        else:
            return None
            
        if not text:
            return None

        # Return transcription instantly when source language same as target language
        if target_lang == origin_lang:
            return {
                "type": "transcription",
                "text": text,
                "start": start_offset,
                "end": start_offset + duration,
                "startClock": startClock
            }

        # Translate based on translation model
        is_en = True if origin_lang == 0 else False
        
        # Use the appropriate translator based on translation_model parameter
        # For now, we only have finetune_translator (mbart)
        # In the future, you can add more translators here
        if translation_model == "mbart" or translation_model == "mbartv2":
            final_text = translate(finetune_translator, text, is_en)
        else:
            # Fallback to default translator
            final_text = translate(finetune_translator, text, is_en)

        # Calculate timestamp
        end_offset = start_offset + duration
        print(f"[ASR] ⏱️ {start_offset:.2f}s -> {end_offset:.2f}s: {final_text}")

        return {
            "type": "transcription",
            "text": final_text,
            "start": start_offset,
            "end": end_offset,
            "startClock": startClock
        }

    except Exception as e:
        print(f"[Inference Error] {e}")
        return None

In [ ]:
# Network + Audio logics
class StreamSession:
    def __init__(self, websocket, vad_model):
        self.ws = websocket
        self.vad_model = vad_model

        # Buffers & VAD State
        self.sentence_buffer = []
        self.silence_counter = 0
        self.is_speaking = False

        # Sync State
        self.anchor_video_time = 0.0
        self.samples_since_anchor = 0
        self.playback_rate = 1.0
        self.sentence_start_video_time = 0.0

        # Metadata
        self.startClock = 0.0
        self.origin_lang = 0
        self.target_lang = 0
        
        # Model Configuration
        self.transcription_model = "wav2vec"
        self.translation_model = "mbart"

    def update_sync(self, data):
        """Handles JSON control messages"""
        if data.get('type') == 'time_sync':
            self.anchor_video_time = float(data['timestamp'])
            self.samples_since_anchor = 0
        elif data.get('type') == 'playback_rate':
            # Update anchor based on how much passed at old speed
            current_offset = (self.samples_since_anchor / VAD_SAMPLE_RATE) * self.playback_rate
            self.anchor_video_time += current_offset
            self.samples_since_anchor = 0
            self.playback_rate = float(data['rate'])
            print(f"[Sync] ⚡ Speed set to {self.playback_rate}x")
        elif data.get('type') == 'config':
            # Update model configuration
            self.transcription_model = data.get('transcriptionModel', 'wav2vec')
            self.translation_model = data.get('translationModel', 'mbart')
            print(f"[Config] 🔧 Transcription: {self.transcription_model}, Translation: {self.translation_model}")

    def parse_audio_message(self, message):
        """Parses binary header and converts audio to Tensor"""
        self.origin_lang = message[0]
        self.target_lang = message[1]
        self.startClock = struct.unpack('<Q', message[2:10])[0]

        audio_data = message[10:]
        audio_int16 = np.frombuffer(audio_data, dtype=np.int16)
        audio_float32 = audio_int16.astype(np.float32) / 32768.0
        return torch.from_numpy(audio_float32)

    def process_vad(self, chunk, current_video_time):
        """Runs VAD on a single chunk and manages the buffer.
           Returns True if a sentence should be triggered."""

        vad_prob = self.vad_model(chunk.unsqueeze(0), VAD_SAMPLE_RATE).item()
        buffer_duration = (len(self.sentence_buffer) * VAD_WINDOW) / VAD_SAMPLE_RATE
        should_trigger = False

        if vad_prob > 0.7:  # Speech
            # Start speaking
            if not self.is_speaking:
                # Get the metadata of start time
                self.sentence_start_video_time = current_video_time

            # Start counter
            self.is_speaking = True
            self.silence_counter = 0
            self.sentence_buffer.append(chunk)

            # Check if buffer has reached max_duration yet
            if buffer_duration >= MAX_SENTENCE_LENGTH:
                should_trigger = True

        else:  # Silence
            # If detect silence while speaking, increase the counter
            if self.is_speaking:
                self.sentence_buffer.append(chunk)
                self.silence_counter += 1

                # If counter reach the limit and buffer has reached the required min sentence length, trigger to cut off sentence
                if self.silence_counter >= SILENCE_CHUNKS and buffer_duration >= MIN_SENTENCE_LENGTH:
                    should_trigger = True

        return should_trigger

    def get_audio_package(self):
        """Prepares the data for inference and resets buffers"""
        full_audio = torch.cat(self.sentence_buffer).numpy()
        speech_duration = (len(full_audio) / VAD_SAMPLE_RATE) * self.playback_rate

        # Snapshot current metadata
        package = {
            "audio": full_audio,
            "start_time": self.sentence_start_video_time,
            "duration": speech_duration,
            "clock": self.startClock,
            "src_lang": self.origin_lang,
            "tgt_lang": self.target_lang,
            "transcription_model": self.transcription_model,
            "translation_model": self.translation_model
        }

        # Reset
        self.sentence_buffer = []
        self.is_speaking = False
        self.silence_counter = 0

        return package

**Processing Thread**

In [ ]:
# ASR + Translate audio
async def ASR_translate_task(websocket, package):
    """Runs inference in thread pool and sends result"""
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(
        None,
        # ASR + Translate cut off audio return by VAD logic
        functools.partial(
            run_inference_sync,
            package['audio'],
            package['start_time'],
            package['duration'],
            package['clock'],
            package['src_lang'],
            package['tgt_lang'],
            package['transcription_model'],
            package['translation_model']
        )
    )
    if result:
        await websocket.send(json.dumps(result))

In [ ]:
# Websocket client's package handler
async def package_handler(websocket):
    print("[WebSocket] Client connected")
    session = StreamSession(websocket, vad_model) # Initialize State

    try:
        async for message in websocket:

            # Control Messages
            if isinstance(message, str):
                try:
                    data = json.loads(message)
                    session.update_sync(data)
                except json.JSONDecodeError:
                    pass
                continue

            # Audio Processing
            audio_tensor = session.parse_audio_message(message)
            number_of_chunks = len(audio_tensor) // VAD_WINDOW

            for i in range(number_of_chunks):
                # Slicing audio into smaller chunk
                start = i * VAD_WINDOW
                end = start + VAD_WINDOW
                chunk = audio_tensor[start:end]

                # Calc Time
                current_time = session.anchor_video_time + \
                    ((session.samples_since_anchor + start) / VAD_SAMPLE_RATE) * session.playback_rate

                # Run VAD Logic
                should_trigger = session.process_vad(chunk, current_time)

                # Fire Inference Task
                if should_trigger:
                    package = session.get_audio_package()
                    asyncio.create_task(ASR_translate_task(websocket, package))

            # Update counter after processing all chunks in this message
            session.samples_since_anchor += len(audio_tensor)

    except websockets.exceptions.ConnectionClosed:
        print("\n[WebSocket] Client disconnected")
    except Exception as e:
        print(f"\n[Error] {e}")

**Main Thread**

In [ ]:
# --- MAIN ENTRY ---
async def main():
    # 1. Load models trước
    load_models()

    # 2. Setup Ngrok
    public_url = ngrok.connect(PORT).public_url
    print(f" * ngrok tunnel \"{public_url}\" -> \"ws://127.0.0.1:{PORT}\"")
    print(f" * CLIENT CONNECT URL: {public_url.replace('https', 'wss').replace('http', 'ws')}")

    # 3. Start Server
    print(f"[Main] Starting WebSocket Server on port {PORT}...")
    async with websockets.serve(package_handler, "localhost", PORT):
        await asyncio.Future()  # Run forever

if __name__ == "__main__":
    # Run
    try:
        asyncio.run(main())
    except RuntimeError as e:
        # Hỗ trợ Colab/Jupyter nếu cần
        if "running event loop" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            asyncio.run(main())
        else:
            raise e